# 🎬 Notebook 3: Item-Based Collaborative Filtering

**Mục tiêu:**
- Tính similarity giữa items (phim) thay vì users
- So sánh với User-Based CF

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from surprise import KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/charts', exist_ok=True)
DEFAULT_K = 40

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

print(f'Loaded: {len(ratings):,} ratings, {len(movies):,} movies')

## 2. Train Item-Based CF

In [ ]:
model = KNNWithMeans(
    k=DEFAULT_K,
    sim_option={'name': 'cosine', 'user_based': False},
    verbose=False
)
model.fit(data.build_full_trainset())
print('✅ Item-Based CF trained!')

In [ ]:
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"\nCV RMSE: {cv_results['test_rmse'].mean():.4f} ± {cv_results['test_rmse'].std():.4f}")

## 3. Gợi ý cho User

In [ ]:
def recommend_item_cf(model, user_id, ratings_df, movies_df, top_n=10):
    """Gợi ý phim bằng Item-Based CF."""
    user_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].values
    all_movies = ratings_df['movieId'].unique()
    unseen = [m for m in all_movies if m not in user_movies]
    scores = {m: model.predict(user_id, m).est for m in unseen}
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    results = []
    for mid, score in sorted_scores[:top_n]:
        title = movies_df[movies_df['movieId'] == mid]['title'].values
        if len(title) > 0:
            results.append({'movieId': mid, 'title': title[0], 'score': round(score, 2)})
    return results

recs = recommend_item_cf(model, 1, ratings, movies, top_n=10)
print('🎬 Top 10 gợi ý cho User 1 (Item-Based CF):')
pd.DataFrame(recs)

## 4. So sánh Item-Based vs User-Based

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

user_cf = KNNWithMeans(k=DEFAULT_K, sim_option={'name': 'cosine', 'user_based': True}, verbose=False)
user_cf.fit(trainset)
user_rmse = accuracy.rmse(user_cf.test(testset), verbose=False)

item_cf = KNNWithMeans(k=DEFAULT_K, sim_option={'name': 'cosine', 'user_based': False}, verbose=False)
item_cf.fit(trainset)
item_rmse = accuracy.rmse(item_cf.test(testset), verbose=False)

print(f'User-Based CF: RMSE = {user_rmse:.4f}')
print(f'Item-Based CF: RMSE = {item_rmse:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['User-Based CF', 'Item-Based CF'], [user_rmse, item_rmse], color=['steelblue', 'coral'], edgecolor='black')
for bar, val in zip(bars, [user_rmse, item_rmse]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.4f}', ha='center', fontweight='bold')
ax.set_ylabel('RMSE')
ax.set_title('User-Based vs Item-Based CF')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/charts/03_user_vs_item_cf.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tổng kết

| Phương pháp | RMSE | Ưu điểm |
|------------|------|--------|
| User-Based | ~0.89 | Tìm niche interests |
| Item-Based | ~0.89 | Nhanh hơn, ổn định |